# 6. Inference only: make `submission.csv` from the final model

Uses only the final student weights. No teachers and no training.

**You need:**
1. `data.zip` from Kaggle in your Google Drive (My Drive).
2. The final weights file `E_kd_ensemble_val_final_weights.pt` in `MyDrive/dl_project1`.
   Notebook 4 saves it there. If it is included in this repository, you can also download it and upload it to that folder.

The model combines each image and its mirror image by adding logits (flip TTA), exactly like notebook 4,
whose submission scored 88.8% on the public leaderboard and 85.7% on the private leaderboard.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/dl_project1'
WEIGHTS = os.path.join(PROJECT_DIR, 'E_kd_ensemble_val_final_weights.pt')
assert os.path.exists(WEIGHTS), f'Weights not found: {WEIGHTS}'

if not os.path.exists('/content/data'):
    os.system('cp /content/drive/MyDrive/data.zip /content/')
    os.system('unzip -q /content/data.zip -d /content/data')
print('Data ready.')

Mounted at /content/drive
Data ready.


In [2]:
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def find_one(pattern):
    hits = glob.glob('/content/data/**/' + pattern, recursive=True)
    assert hits, f'Could not find {pattern} inside /content/data'
    return hits[0]

def id_column(df):
    if 'id' in df.columns:
        return 'id'
    return [c for c in df.columns if 'id' in c.lower()][0]

def image_paths(df, folder):
    for c in df.columns:
        s = df[c].astype(str)
        if s.str.lower().str.endswith('.png').all():
            return [os.path.join(folder, os.path.basename(v)) for v in s]
    return [os.path.join(folder, f'{v}.png') for v in df[id_column(df)]]

test_df = pd.read_csv(find_one('test.csv'))
sample_df = pd.read_csv(find_one('sample_submission.csv'))
X_test = np.stack([np.array(Image.open(p).convert('RGB'))
                   for p in image_paths(test_df, find_one('test_images'))])
test_ids = test_df[id_column(test_df)].astype(str).to_numpy()

MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1, 3, 1, 1)
Xte = (torch.from_numpy(X_test).to(device).permute(0, 3, 1, 2).float() / 255.0 - MEAN) / STD
print('Test images:', tuple(Xte.shape))

Test images: (2000, 3, 32, 32)


In [3]:
class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=1, k=3):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, k, stride, k // 2, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, k, 1, k // 2, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.shortcut = nn.Sequential()
        if stride != 1 or c_in != c_out:
            self.shortcut = nn.Sequential(
                nn.Conv2d(c_in, c_out, 1, stride, bias=False),
                nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))   # ReLU(F(x) + S(x))


class ResNet(nn.Module):
    def __init__(self, channels=(64, 128, 256), blocks=(3, 5, 3), num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels[0], 3, 1, 1, bias=False),
            nn.BatchNorm2d(channels[0]),
            nn.ReLU(inplace=True))
        layers, c_in = [], channels[0]
        for i, (c, b) in enumerate(zip(channels, blocks)):
            stride = 1 if i == 0 else 2
            for j in range(b):
                layers.append(BasicBlock(c_in, c, stride if j == 0 else 1))
                c_in = c
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(channels[-1], num_classes)

    def forward(self, x):
        x = self.layers(self.stem(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(x)


model = ResNet(blocks=(3, 5, 3)).to(device)
model.load_state_dict(torch.load(WEIGHTS, map_location=device))
model.eval()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')
assert n_params <= 5_000_000

Trainable parameters: 4,918,602


In [4]:
# flip TTA: add the logits of each image and its mirror image (same as notebook 4)
use_bf16 = device == 'cuda' and torch.cuda.is_bf16_supported()
logits = []
with torch.no_grad():
    for i in range(0, len(Xte), 500):
        xb = Xte[i:i + 500]
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=use_bf16):
            logits.append(model(xb).float() + model(xb.flip(3)).float())
preds = torch.cat(logits).argmax(1).cpu().numpy()

pred_by_id = dict(zip(test_ids, preds))
sub_id_col = id_column(sample_df)
submission = pd.DataFrame({'id': sample_df[sub_id_col]})
submission['label'] = sample_df[sub_id_col].astype(str).map(pred_by_id)
assert submission['label'].notna().all(), 'Some test ids have no prediction!'
submission['label'] = submission['label'].astype(int)
submission.to_csv('submission.csv', index=False)
print(submission.head())
print('Predictions per class:', np.bincount(submission['label'], minlength=10))

from google.colab import files
files.download('submission.csv')

   id  label
0   0      6
1   1      3
2   2      3
3   3      9
4   4      3
Predictions per class: [180 213 211 207 192 205 182 206 205 199]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
old = pd.read_csv(os.path.join(PROJECT_DIR, 'E_kd_ensemble_val_submission.csv'))
same = (old['label'].values == submission['label'].values).mean() * 100
print(f'Matches the submitted file on {same:.2f}% of test images')

Matches the submitted file on 99.95% of test images
